In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# 设置绘图风格与中文字体
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 DWD 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_orientation_data(city_name):
    """
    获取指定城市的朝向分布数据。
    """
    conn = get_db_connection()
    query = f"""
    SELECT 
        orientation,
        COUNT(house_id) as house_count
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city = '{city_name}' 
      AND orientation IS NOT NULL 
      AND orientation != ''
      AND orientation != '未知'
    GROUP BY orientation
    ORDER BY house_count DESC
    LIMIT 12
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_nightingale_rose(city):
    """绘制南丁格尔玫瑰图 (极坐标柱状图)"""
    df = fetch_orientation_data(city)
    
    if df.empty or len(df) < 3:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 朝向数据不足，无法生成玫瑰图", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    # --- 1. 数据准备与极坐标参数 ---
    labels = df['orientation'].tolist()
    counts = df['house_count'].values
    num_vars = len(labels)

    # 计算每个柱子在极坐标中的角度位置 (均分 360 度)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False)
    # 计算每个柱子的宽度
    width = 2 * np.pi / num_vars

    # --- 2. 颜色映射 ---
    # 数量越多的朝向，颜色越鲜艳
    norm = mcolors.Normalize(vmin=counts.min(), vmax=counts.max())
    # 使用 'Spectral_r' 调色板，呈现出花瓣渐变色
    cmap = cm.get_cmap('Spectral_r') 
    colors = cmap(norm(counts))

    # --- 3. 开始绘图 (引入极坐标系) ---
    fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

    ax.set_theta_zero_location("N")  
    ax.set_theta_direction(-1)       

    # 绘制极坐标柱状图
    bars = ax.bar(
        angles, counts, 
        width=width, 
        bottom=counts.max() * 0.1,  
        color=colors, 
        edgecolor='white', 
        linewidth=1.5,
        alpha=0.85
    )

    # --- 4. 标注与修饰 ---
    ax.set_xticks(angles)
    ax.set_xticklabels(labels, fontsize=12, fontweight='bold', color='#2C3E50')

    # 在每个花瓣的顶端标注具体的房源数量
    for angle, count, bar in zip(angles, counts, bars):
        rotation = np.degrees(angle)
        if 90 < rotation < 270:
            rotation += 180 
            
        ax.text(
            angle, 
            count + (counts.max() * 0.2), 
            f'{count:,}', 
            ha='center', va='center', 
            fontsize=10, 
            color='#555555',
            rotation=rotation,
            rotation_mode='anchor'
        )

    # 图表标题与辅助线设置
    plt.title(f'[{city}] 房源朝向分布特征 (南丁格尔玫瑰图)', fontsize=18, pad=40, fontweight='bold')
    
    # 隐藏径向的 Y 轴刻度标签，让图形更加纯粹
    ax.set_yticklabels([])
    # 弱化网格线
    ax.grid(color='#E5E7E9', linestyle='--', linewidth=1)
    
    # 移除极坐标最外圈的黑色圆环线
    ax.spines['polar'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [12]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_nightingale_rose(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_nightingale_rose(default_city)

In [13]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京', …

Output()